# Week 1 · Notebook 6 — Dip-Buy Strategy (TikTok Strategy)

This notebook builds a **mean-reversion dip-buy strategy** from scratch.

The rule is simple:
- **Buy** any stock that drops **5% or more** in a single day.
- **Sell** once that position gains **10%** from your entry price.
- **Hold** indefinitely until the 10% target is reached — no stop-loss, no time limit.
- **Size** — split all available cash equally across every stock that triggers a buy on the same day.

What you will build:
1. Understand why this is a *mean-reversion* idea.
2. Inspect real dip events in the data.
3. Write `DipBuyStrategy` — a stateful callable class.
4. Run a full backtest and compare to the benchmarks.
5. Graduate the class into `src/tradinglab/strategies/dip_buy.py`.

## 1. Setup

In [ ]:
import sys, os
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from tradinglab.data_feed import DataFeed
from tradinglab.simulator import PortfolioSimulator
from tradinglab.backtester import run_backtest
from tradinglab.observation import build_observation
from tradinglab.metrics import total_return, sharpe, max_drawdown

feed = DataFeed.from_dir('data/egx')   # all 34 EGX assets
print(f'Universe : {feed.n_assets} assets')
print(f'Date range: {feed.dates[0].date()} → {feed.dates[-1].date()}')
print(f'Trading days: {feed.n_days}')

## 2. The idea — mean reversion

**Trend-following** (like SMA crossover) bets that what went up keeps going up.
**Mean reversion** bets the opposite: after an unusually large move, the price
tends to bounce back toward its average.

The dip-buy rule is one of the oldest expressions of mean reversion:
> "This stock fell hard today — probably an overreaction. Buy the dip, wait for the bounce."

The 5% / 10% numbers are thresholds:
- **5% drop** — filters out normal daily noise, targets genuine panic selling.
- **10% gain** — a realistic bounce target; takes profit before the stock might reverse again.

Let's first check: how often do 5%+ drops actually happen in our universe?

In [ ]:
DIP_THRESHOLD  = -0.05   # -5%
TAKE_PROFIT    =  0.10   # +10%

# Count dip events across the whole universe
dip_mask = feed.returns <= DIP_THRESHOLD   # shape (n_days, n_assets)
total_dips = int(dip_mask.sum())
total_obs  = feed.n_days * feed.n_assets

print(f'Total observations  : {total_obs:,}')
print(f'Dip events (≤ -5%)  : {total_dips:,}  ({100*total_dips/total_obs:.2f}% of all days)')
print(f'Average dips per day: {total_dips / feed.n_days:.2f}')
print()

# Which stock dips the most?
print('Dips per stock:')
for i, sym in enumerate(feed.symbols):
    n = int(dip_mask[:, i].sum())
    print(f'  {sym:6s}  {n:4d}  ({100*n/feed.n_days:.1f}% of trading days)')

## 3. Visualise a dip and its recovery on a single stock

Pick one stock and highlight every day it dipped 5%+. This builds intuition
before we write any strategy code.

In [ ]:
SYMBOL = 'COMI'
idx    = feed.symbols.index(SYMBOL)

prices = feed.close[:, idx]
returns = feed.returns[:, idx]
dip_days = np.where(returns <= DIP_THRESHOLD)[0]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Price with dip markers
ax1.plot(feed.dates, prices, color='#4da3ff', linewidth=1.2, label=SYMBOL + ' close')
ax1.scatter(feed.dates[dip_days], prices[dip_days],
            color='#ff4d6a', s=40, zorder=5, label='≤ -5% dip')
ax1.set_ylabel('Price (EGP)')
ax1.legend(); ax1.grid(alpha=0.25)
ax1.set_title(f'{SYMBOL} — price and dip events')

# Daily return
ax2.bar(feed.dates, returns,
        color=['#ff4d6a' if r <= DIP_THRESHOLD else '#26c466' if r > 0 else '#8b9baf'
               for r in returns],
        width=1)
ax2.axhline(DIP_THRESHOLD, color='#ff4d6a', linestyle='--', linewidth=0.8, label=f'{DIP_THRESHOLD:.0%} threshold')
ax2.set_ylabel('Daily return')
ax2.legend(); ax2.grid(alpha=0.25)

plt.tight_layout()
plt.show()

print(f'{SYMBOL} had {len(dip_days)} dip days out of {feed.n_days} trading days')

## 4. How often does the bounce actually happen?

Before committing to the strategy, check the historical hit rate:
after a 5% dip, how often did the stock eventually reach +10% from the entry price?

In [ ]:
hits = 0
total = 0
days_to_hit = []

for asset_idx in range(feed.n_assets):
    asset_returns = feed.returns[:, asset_idx]
    for t in range(1, feed.n_days - 1):
        if asset_returns[t] <= DIP_THRESHOLD:
            total += 1
            # Check forward: does it hit +10% from this entry?
            cum = 1.0
            for future in range(t + 1, feed.n_days):
                cum *= (1 + asset_returns[future])
                if cum >= 1 + TAKE_PROFIT:
                    hits += 1
                    days_to_hit.append(future - t)
                    break

hit_rate = hits / total if total > 0 else 0
print(f'Total dip events analysed : {total}')
print(f'Hit take-profit (+10%)     : {hits}  ({hit_rate:.1%})')
print(f'Did NOT hit take-profit    : {total - hits}  ({1-hit_rate:.1%})')
if days_to_hit:
    print(f'Median days to hit +10%    : {int(np.median(days_to_hit))}')
    print(f'Mean days to hit +10%      : {np.mean(days_to_hit):.1f}')

## 5. Write `DipBuyStrategy`

Unlike SMA crossover (a pure function), this strategy needs **memory** — it must
remember which stocks it already holds and at what entry price. We use a
**callable class** for this.

The backtester calls it identically to a plain function: `strategy(observation) -> weights`.

**Internal state (three arrays, one value per asset):**
- `_weights[i]` — current portfolio weight in asset i (0 = not held)
- `_entry_price[i]` — normalised price at entry (always 1.0 when set)
- `_current_price[i]` — price today relative to entry (tracks the gain)

**Each day, in order:**
1. Update `_current_price` for every held position using today's return.
2. Sell any position that reached +10% (weight → 0, free up cash).
3. Find new dips (drop ≤ -5% today, not already held).
4. Split available cash equally across all new dips.

In [ ]:
class DipBuyStrategy:
    """Stateful dip-buy / mean-reversion strategy.

    BUY  — stock dropped >= dip_threshold today, not already held.
    SELL — position gained >= take_profit from entry price.
    SIZE — available cash split equally across all new buy signals.

    Parameters
    ----------
    dip_threshold : float  (default -0.05 = -5%)
    take_profit   : float  (default  0.10 = +10%)
    """

    def __init__(self, dip_threshold=-0.05, take_profit=0.10):
        self.dip_threshold = dip_threshold
        self.take_profit   = take_profit
        self._weights       = None
        self._entry_price   = None
        self._current_price = None

    def reset(self, n_assets):
        """Initialise (or reset) internal state for a new backtest."""
        self._weights       = np.zeros(n_assets)
        self._entry_price   = np.zeros(n_assets)
        self._current_price = np.zeros(n_assets)

    def __call__(self, observation):
        """
        observation : (n_assets, lookback, n_features)
        returns     : (n_assets,) weight vector, non-negative, sums to <= 1
        """
        n_assets = observation.shape[0]
        if self._weights is None:
            self.reset(n_assets)

        weights       = self._weights.copy()
        entry_price   = self._entry_price.copy()
        current_price = self._current_price.copy()

        # Feature 0 of the observation is the daily return for each asset.
        today_return = observation[:, -1, 0]   # (n_assets,)

        # ── Step 1: update current prices for all held positions ──────────
        for i in range(n_assets):
            if weights[i] > 0:
                current_price[i] *= (1.0 + today_return[i])

        # ── Step 2: take profit — sell anything up 10% from entry ─────────
        for i in range(n_assets):
            if weights[i] > 0:
                gain = (current_price[i] / entry_price[i]) - 1.0
                if gain >= self.take_profit:
                    weights[i]       = 0.0
                    entry_price[i]   = 0.0
                    current_price[i] = 0.0

        # ── Step 3: find new dips (dropped today, not already held) ───────
        new_buys = [
            i for i in range(n_assets)
            if today_return[i] <= self.dip_threshold and weights[i] == 0.0
        ]

        # ── Step 4: allocate available cash equally to new buys ───────────
        if new_buys:
            cash       = max(0.0, 1.0 - weights.sum())   # uninvested fraction
            alloc_each = cash / len(new_buys)             # equal split
            for i in new_buys:
                if alloc_each > 0:
                    weights[i]       = alloc_each
                    entry_price[i]   = 1.0
                    current_price[i] = 1.0 + today_return[i]

        # ── Persist state ─────────────────────────────────────────────────
        self._weights       = weights
        self._entry_price   = entry_price
        self._current_price = current_price

        return weights


# ── Quick sanity check ────────────────────────────────────────────────────────
strat = DipBuyStrategy()
obs   = build_observation(feed, 100, lookback=1)
w     = strat(obs)

assert w.shape == (feed.n_assets,), 'wrong shape'
assert (w >= 0).all(), 'negative weights'
assert w.sum() <= 1.0 + 1e-9, 'weights exceed 1'
print('DipBuyStrategy sanity check ✓')
print(f'Weights sum on day 100: {w.sum():.4f}')
print('Positions opened:', [feed.symbols[i] for i in range(feed.n_assets) if w[i] > 0])

## 6. Step through a few days manually

Trace exactly what happens in the first 10 days so there are no surprises when
the full backtest runs.

In [ ]:
strat = DipBuyStrategy()
START = 30
print(f'{"day":>5s}  {"dips":>6s}  {"n_held":>6s}  {"cash%":>6s}  {"symbols held"}')

for t in range(START, START + 20):
    obs    = build_observation(feed, t, lookback=1)
    w      = strat(obs)
    n_held = int((w > 0).sum())
    today_ret = obs[:, -1, 0]
    n_dips = int((today_ret <= DIP_THRESHOLD).sum())
    cash   = max(0.0, 1.0 - w.sum())
    held   = [feed.symbols[i] for i in range(feed.n_assets) if w[i] > 0]
    print(f'{t:5d}  {n_dips:6d}  {n_held:6d}  {cash:6.1%}  {held}')

## 7. Full backtest — strategy vs benchmarks

Run the full backtest with a 0.5% commission (50 bps per unit of turnover),
the same setting used in the dashboard.

In [ ]:
COMMISSION   = 0.005
START_CAPITAL = 1000   # EGP

# Strategy vs EGX30 benchmark
sim      = PortfolioSimulator(feed, benchmark='egx30', commission=COMMISSION)
strategy = DipBuyStrategy()
result   = run_backtest(sim, strategy, lookback=1)

# Equal-weight benchmark (separate run so benchmarks are independent)
sim_eq      = PortfolioSimulator(feed, benchmark='equal_weight', commission=COMMISSION)
strategy_eq = DipBuyStrategy()
result_eq   = run_backtest(sim_eq, strategy_eq, lookback=1)

dates     = result['dates']
portfolio = result['portfolio'] * START_CAPITAL
bench_egx = result['benchmark'] * START_CAPITAL
bench_eq  = result_eq['benchmark'] * START_CAPITAL

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(dates, portfolio,  color='#26c466', linewidth=2,   label='Dip-Buy strategy')
ax.plot(dates, bench_egx,  color='#ff4d6a', linewidth=1.5, label='Benchmark — EGX30')
ax.plot(dates, bench_eq,   color='#a78bfa', linewidth=1.5,
        linestyle='--', label='Benchmark — Equal Weight')
ax.set_ylabel('Portfolio value (EGP, started at 1 000)')
ax.set_title('Dip-Buy Strategy vs Benchmarks  |  commission = 0.5%')
ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 8. Performance metrics

In [ ]:
def report(label, equity_curve):
    rets = np.diff(equity_curve) / equity_curve[:-1]
    tr   = total_return(rets)
    sh   = sharpe(rets)
    md   = max_drawdown(rets)
    final = equity_curve[-1]
    print(f'{label:<30s}  final={final:8.1f} EGP  '
          f'total_return={tr:+.1%}  sharpe={sh:.2f}  max_dd={md:.1%}')

report('Dip-Buy (with commission)',  portfolio)
report('Benchmark — EGX30',          bench_egx)
report('Benchmark — Equal Weight',   bench_eq)

## 9. Compare to SMA crossover side by side

In [ ]:
from tradinglab.strategies.sma import sma_crossover_weights

sim_sma    = PortfolioSimulator(feed, benchmark='egx30', commission=COMMISSION)
result_sma = run_backtest(sim_sma, sma_crossover_weights, lookback=30)
sma_port   = result_sma['portfolio'] * START_CAPITAL

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(dates,                portfolio, color='#26c466', linewidth=2,   label='Dip-Buy')
ax.plot(result_sma['dates'],  sma_port,  color='#4da3ff', linewidth=2,   label='SMA Crossover')
ax.plot(dates,                bench_egx, color='#ff4d6a', linewidth=1.5, label='EGX30')
ax.plot(dates,                bench_eq,  color='#a78bfa', linewidth=1.5,
        linestyle='--', label='Equal Weight')
ax.set_ylabel('Portfolio value (EGP, started at 1 000)')
ax.set_title('Dip-Buy vs SMA Crossover vs Benchmarks  |  commission = 0.5%')
ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print()
report('Dip-Buy',               portfolio)
report('SMA Crossover',         sma_port)
report('Benchmark — EGX30',     bench_egx)
report('Benchmark — Equal Wt',  bench_eq)

## 10. What the strategy is actually doing — position analysis

Let's look at how many positions are held each day and how invested we typically are.

In [ ]:
# Re-run strategy and record weights at each step
strat2 = DipBuyStrategy()
weights_history = []
for t in range(1, feed.n_days - 1):
    obs = build_observation(feed, t, lookback=1)
    w   = strat2(obs)
    weights_history.append(w.copy())

weights_history = np.array(weights_history)   # (n_days-2, n_assets)

n_positions  = (weights_history > 0).sum(axis=1)
invested_pct = weights_history.sum(axis=1)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax1.plot(feed.dates[1:-1], n_positions, color='#4da3ff', linewidth=0.8)
ax1.set_ylabel('# positions held')
ax1.set_title('Daily position count and cash allocation')
ax1.grid(alpha=0.25)

ax2.fill_between(feed.dates[1:-1], invested_pct, color='#26c466', alpha=0.4, label='invested')
ax2.fill_between(feed.dates[1:-1], 1, invested_pct, color='#8b9baf', alpha=0.2, label='cash')
ax2.set_ylabel('Fraction of portfolio')
ax2.set_ylim(0, 1.05)
ax2.legend(); ax2.grid(alpha=0.25)

plt.tight_layout()
plt.show()

print(f'Average positions held per day : {n_positions.mean():.1f}')
print(f'Max positions on a single day  : {n_positions.max()}')
print(f'Days fully in cash             : {(n_positions == 0).sum()}')
print(f'Average % invested             : {invested_pct.mean():.1%}')

## 11. Sensitivity — what if we change the thresholds?

Is 5%/10% the best choice, or just the one we started with? A quick grid search
shows how Sharpe changes with different dip and take-profit levels.

In [ ]:
dip_levels = [-0.03, -0.05, -0.07, -0.10]
tp_levels  = [0.05,   0.10,  0.15,  0.20]

print(f'{"dip":>6s}  {"TP":>5s}  {"total_ret":>10s}  {"sharpe":>7s}  {"max_dd":>8s}')
for dip in dip_levels:
    for tp in tp_levels:
        s  = PortfolioSimulator(feed, benchmark='egx30', commission=COMMISSION)
        st = DipBuyStrategy(dip_threshold=dip, take_profit=tp)
        r  = run_backtest(s, st, lookback=1)
        eq = r['portfolio'] * START_CAPITAL
        rets = np.diff(eq) / eq[:-1]
        marker = ' <-- original' if dip == -0.05 and tp == 0.10 else ''
        print(f'{dip:6.0%}  {tp:5.0%}  {total_return(rets):10.1%}  '
              f'{sharpe(rets):7.2f}  {max_drawdown(rets):8.1%}{marker}')

## 12. Graduate

The `DipBuyStrategy` class already lives in:

```
src/tradinglab/strategies/dip_buy.py
```

It is imported and used live in the dashboard at `/backtest/dip`.
You can switch to it in the browser using the **Strategy** dropdown in the equity panel.

### What you learned

| Concept | Where it showed up |
|---|---|
| Mean reversion | The core thesis: dips tend to bounce |
| Stateful strategy | Needed a class, not a plain function |
| Cash management | Available cash split equally — no leverage |
| Commission drag | 0.5% per turnover hurts high-frequency strategies |
| Sensitivity analysis | 5%/10% is a starting point, not the answer |

### Ideas to try next

- Add a **stop-loss**: cut the position if it falls another 5% from entry.
- **Volume filter**: only buy the dip if volume is above average (real selling, not a thin-market quirk).
- **Sector diversification**: limit exposure to any one sector.
- Feed the dip-buy signal into the **RL agent** as a feature in week 3.